---
title: "Lab 11: Secuencias"
author: "Maximiliano Garnier Villarreal"
lang: es
toc: true
toc-depth: 3
toc-title: Contenidos
number-sections: true
highlight-style: pygments
theme: sandstone
format:
  html:
    embed-resources: true
    code-fold: true
    code-summary: "Codigo"
    code-tools: true
    html-math-method: katex
  pdf: 
    prefer-html: false
  docx: default
execute:
  warning: false
  error: false
  echo: true
---

# Paquetes

In [ ]:
#| label: setup
#| include: false

import numpy as np
import pandas as pd
import polars as pl
from scipy import stats
import pingouin as pg
import plotnine as p9 
import matplotlib.pyplot as plt
import marginaleffects as me
import statsmodels.formula.api as smf
import statsmodels.api as sm
import statsmodels.stats.api as sms
from statsmodels.tools.eval_measures import rmse
from statsmodels.stats.anova import anova_lm

p9.theme_set(p9.theme_minimal(base_size = 14))

# Prueba de tendencia (Spearman)

Se determina si la longitud de los intervalos varia en el tiempo/espacio

$$H_0 : \text{longitud de intervalos es igual}$$

In [ ]:
a = .05

x = np.array([0.5,2.3,3.2,4.2,4.9,7,11.4,12.7,14.6,
      16,21.5,22.5,25.8,30.3,31.9,36.2,42.8])
x.sort()

n = len(x)
i = np.arange(1,n,1)
n_i = len(i)
h = np.empty(n-1)

for j in range(1,n):
  h[j-1] = x[j]-x[j-1]

rankh = stats.rankdata(h, method="average")
cor_dat = pd.DataFrame({'i': i, 'h': h, 'rankh': rankh})

rs = 1-(6*sum((i-rankh)**2))/(n_i*(n_i**2-1))

print(f"r_spearman: {rs:.3f}")

In [ ]:
print(pg.corr(i, h, method='spearman'))

In [ ]:
rs = stats.pearsonr(i, rankh)
rs_ci = stats.pearsonr(i, rankh).confidence_interval(confidence_level=1-a)

print(f"r: {rs[0]:.3f}, IC {(1-a)*100:.0f}% ({rs_ci[0]:.3f}, {rs_ci[1]:.3f}), valor-p: {rs[1]:.3f}")

# Prueba de corridas

Se determina si la secuencia, dicotomica, es aleatoria o no. Si se tienen datos continuos se tienen que dicotomizar, se pueden usar los criterios con respecto a la mediana (`median`) o arriba-abajo (`up-down`).

$$H_0 : \text{secuencia es aleatoria}$$

In [ ]:
dens = np.array([3.57,3.63,2.86,2.94,3.42,2.85,3.67,3.78,3.86,
         4.02,4.56,4.62,4.31,4.58,5.02,4.68,4.37,4.88,
         4.52,4.80,4.55,4.62,4.93,4.60,4.51,3.98,4.22,
         3.52,2.91,3.87,3.52,3.77,3.84,3.92,4.09,3.86,
         4.13,3.92,3.54])

corridas = sms.runstest_1samp(dens, cutoff='median', correction=False) # test de corridas

print(f"Z: {corridas[0]:.3f}, valor-p: {corridas[1]:.3f}")

Si se tiene una secuencia dicotómica, se puede usar el test de corridas, pero primero hay que codificarla como datos binarios.

In [ ]:
data = ['P', 'F', 'P', 'P', 'F', 'F', 'P', 'F', 'P', 'P', 'P', 'F']
binary_data = [1 if x == 'P' else 0 for x in data]

corridas = sms.runstest_1samp(binary_data, correction=False)

print(f"Z: {corridas[0]:.3f}, valor-p: {corridas[1]:.3f}")
print(pd.Series(data).value_counts())

Funcion para calcular el test de corridas arriba-abajo.

In [ ]:
from fns.gmisc import up_down_runs_test

In [ ]:
up_down_runs_test(dens)

# Regresion

La idea es determinar si un modelo es significativo o no, o sea si los coeficientes (especialmente la pendiente $\hat{b}_1$) son significativos

$$H_0 : \hat{b}_1 = 0$$

En el caso de un modelo mas complejo de uno lineal la idea es determinar si la adicion de los nuevos terminos es significativa y explica (se ajusta) mejor (a) los datos

$$H_0 : \hat{b}_n = 0$$

In [ ]:
alfa = 0.05

depth = np.arange(1,41,5) # profundidad
moist = np.array([124,78,54,35,30,21,22,18]) # contenido de humedad

DF3 = pd.DataFrame({'depth': depth, 'moist': moist})

## Grafico

In [ ]:
(p9.ggplot(DF3, p9.aes('depth','moist')) + 
  p9.geom_point())

## Modelo lineal

In [ ]:
fit1 = smf.ols('moist ~ depth', data=DF3).fit()

print(fit1.summary(alpha=alfa))
# print(fit1.conf_int(alpha=alfa))

## Modelo cuadratico

In [ ]:
fit2 = smf.ols('moist ~ depth + I(depth ** 2)', data=DF3).fit()

print(fit2.summary(alpha=alfa))
# print(fit2.conf_int(alpha=alfa))

## Modelo logaritmico

In [ ]:
fit3 = smf.ols('moist ~ np.log(depth)', data=DF3).fit()

print(fit3.summary(alpha=alfa))
# print(fit3.conf_int(alpha=alfa))

In [ ]:
# intervalos de predicción para nuevos datos
new_X = pd.DataFrame({'depth': np.linspace(1, 40, 50)})
fit3_pred = fit3.get_prediction(new_X).summary_frame(alpha=alfa)
print(fit3_pred.head())

## Grafico con modelos

In [ ]:
(p9.ggplot(DF3, p9.aes("depth", "moist")) +
  p9.geom_point(size=3) +
  p9.geom_smooth(p9.aes(color='"Lin"', fill='"Lin"'), 
                 method='lm', se=False, alpha=0.2) +
  p9.geom_smooth(p9.aes(color='"Cuad"', fill='"Cuad"'), 
                 method='lm', se=False, alpha=0.2,
                 formula='y ~ x + I(x**2)') +
  p9.geom_smooth(p9.aes(color='"Log"', fill='"Log"'), 
                 method='lm', se=False, alpha=0.2,
                 formula='y ~ np.log(x)') +
  p9.scale_color_brewer(type='qual', palette=2, name='Modelo') +
  p9.scale_fill_brewer(type='qual', palette=2, name='Modelo'))

In [ ]:
me.plot_predictions(fit1, condition='depth', points=1)

In [ ]:
me.plot_predictions(fit2, condition='depth', points=1)

In [ ]:
me.plot_predictions(fit3, condition='depth', points=1)

## Comparacion de modelos

In [ ]:
print(anova_lm(fit1,fit2))
print(anova_lm(fit3,fit2))

In [ ]:
mods = [fit1, fit2, fit3]
mods_dict = {"Lin": fit1, "Cuad": fit2, "Log": fit3}

# rmse_vals = [rmse(mod.fittedvalues, DF3['moist']) for mod in mods]

rows = []
for name, m in mods_dict.items():
    resid = getattr(m, "resid", None)
    rows.append({
        "model": name,
        "AIC": getattr(m, "aic", np.nan),
        "BIC": getattr(m, "bic", np.nan),
        "llf": getattr(m, "llf", np.nan),
        "R2": getattr(m, "rsquared", np.nan),
        "adj_R2": getattr(m, "rsquared_adj", np.nan),
        "RMSE": rmse(m.model.endog, m.fittedvalues) if (hasattr(m, "fittedvalues") and hasattr(m.model, "endog")) else np.nan,
        "MAE": np.mean(np.abs(resid)) if resid is not None else np.nan,
        "F": getattr(m, "fvalue", np.nan),
        "F_pvalue": getattr(m, "f_pvalue", np.nan),
        # "df_model": int(getattr(m, "df_model", np.nan)) if getattr(m, "df_model", None) is not None else np.nan,
        # "df_resid": int(getattr(m, "df_resid", np.nan)) if getattr(m, "df_resid", None) is not None else np.nan
    })

df_metrics = pd.DataFrame(rows).set_index("model")
df_metrics = df_metrics.round(4)

df_metrics

In [ ]:
# Transponer y limpiar etiquetas
df_metrics_T = df_metrics.T
df_metrics_T.index.name = "statistic"
df_metrics_T = df_metrics_T.round(4)

# devuelve el DataFrame transpuesto
df_metrics_T

# Autocorrelacion

Determina si hay ciclos, patrones, tendencia, sobre una serie de datos ordenados, preferiblemente equidistantemente, en el espacio/tiempo.

In [ ]:
from statsmodels.tsa.stattools import acf, pacf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.stats.stattools import durbin_watson

## Visualizar los datos

In [ ]:
airpassengers = pd.read_csv('data/airpassengers.csv')

**Para este caso**, se considera que el índice es una combinación del anho y el mes, y se crean las columnas correspondientes.

In [ ]:
df = airpassengers.copy()  # asume DataFrame con columnas 'index' (año.fracción) y 'value'
df['year'] = df['index'].astype(int)
df['month'] = (np.rint((df['index'] - df['year']) * 12).astype(int) + 1).clip(1,12)
df['date'] = pd.to_datetime(dict(year=df['year'], month=df['month'], day=1))
df['log_value'] = np.log(df['value'])
df['value_diff'] = df['value'].diff()

In [ ]:
(p9.ggplot(df, p9.aes(x='date', y='value')) + 
  p9.geom_line())

In [ ]:
#| eval: false

import plotly.express as px

fig = px.line(df, x='date', y='value',
              title='AirPassengers',
              labels={'value':'Passengers', 'date':'Date'},
              markers=False)
fig.update_xaxes(dtick="M12", tickformat="%Y", rangeslider_visible=True)
fig.update_layout(template='plotly_white')
fig

## Como remover una tendencia de una serie de datos

Pasos:

1.  Determinar si hay una tendencia

    a. Visualizar los datos 
    b. Construir modelo lineal
    c. Determinar si la pendiente es significativa

2.  Si hay tendencia, calcular los residuos

### Visualizar datos

In [ ]:
(p9.ggplot(df, p9.aes(x='index', y='log_value')) + 
  p9.geom_line() +
  p9.geom_smooth(method = 'lm'))

### Modelo y residuos

In [ ]:
mod = smf.ols('log_value ~ index', data=df).fit()

print(mod.summary().tables[1])

In [ ]:
df2 = df.copy()
df2['.resid'] = mod.resid

### Visualizar residuales

In [ ]:
(p9.ggplot(df2, p9.aes(x='index', y='.resid')) +
  p9.geom_line() +
  p9.geom_hline(yintercept=0, color='red'))

## Grafico de autocorrelacion

In [ ]:
n = len(df2)
lag_max = int(n/3)
delta = np.diff(df2['index'])[0]

### Sin remover tendencia

In [ ]:
plot_acf(df2['value'], lags=lag_max, alpha=alfa, fft=True, bartlett_confint=False, auto_ylims=True)

### Despues de remover tendencia

In [ ]:
plot_acf(df2['.resid'], lags=lag_max, alpha=alfa, fft=True, bartlett_confint=False, auto_ylims=True)

## Prueba de autocorrelacion

### Objeto dataframe

In [ ]:
acf_vals = acf(df2['.resid'], nlags=lag_max, fft=True)

acf_df = pd.DataFrame({
  'lag': np.arange(len(acf_vals)), 
  'ACF': acf_vals
  })
acf_df

In [ ]:
lag_t = acf_df[acf_df['ACF']<0]

start_lag = lag_t.iloc[0, 0]
start_idx = acf_df.index[acf_df['lag'] == start_lag][0]
rtau_1 = acf_df.loc[start_idx:, 'ACF'].max() # valor maximo de autocorrelacion

tau_1 = acf_df['lag'][acf_df['ACF'] == rtau_1].values # lag donde ocurre el valor maximo de autocorrelacion

z_1 = rtau_1 * np.sqrt(n - tau_1 + 3) # test de significancia de autocorrelacion

p_ac = stats.norm.cdf(-abs(z_1)) * 2 # valor-p

ac_dist = tau_1 * delta # distancia de autocorrelacion

ac_res = pd.DataFrame({
  'estimate': rtau_1,
  'lag': tau_1,
  'delta': delta,
  'dist': ac_dist,
  'statistic': z_1,
  'p.value': p_ac
}
)

ac_res